In [ ]:
!uv init

In [ ]:
!uv add relbench torch torch-geometric pandas tqdm matplotlib seaborn torch -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

In [55]:
class LightGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels=64, out_channels=32, dropout=0.2):
        super(LightGNN, self).__init__()

        self.dropout = dropout

        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

        self.bn1 = nn.BatchNorm1d(hidden_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.normalize(x, p=2, dim=-1)

        return x

    def decode(self, z, edge_index):
        return (z[edge_index[0]] * z[edge_index[1]]).sum(dim=-1)

In [ ]:
class Trainer:
    def __init__(
        self,
        model,
        data,
        batch_size=2048,
        device="cuda" if torch.cuda.is_available() else "cpu",
    ):

        self.model = model.to(device)
        self.device = device
        self.batch_size = batch_size

        self.x = data.x.to(device)
        self.edge_index = data.edge_index.to(device)

        self.val_edges = data.val_edges
        self.test_edges = data.test_edges
        self.num_nodes = data.num_nodes

    def train_epoch(self, optimizer):
        self.model.train()
        total_loss = 0
        num_batches = 0

        num_train_edges = self.edge_index.size(1)
        num_samples = min(100000, num_train_edges)

        perm = torch.randperm(num_train_edges)[:num_samples]
        sampled_edges = self.edge_index[:, perm]

        for i in range(0, num_samples, self.batch_size):
            batch_edges = sampled_edges[:, i : i + self.batch_size]
            if batch_edges.size(1) == 0:
                continue

            optimizer.zero_grad()

            z = self.model(self.x, self.edge_index)

            pos_pred = self.model.decode(z, batch_edges)

            neg_edges = negative_sampling(
                edge_index=batch_edges,
                num_nodes=self.num_nodes,
                num_neg_samples=batch_edges.size(1),
            )
            neg_pred = self.model.decode(z, neg_edges)

            pos_loss = F.binary_cross_entropy_with_logits(
                pos_pred, torch.ones_like(pos_pred)
            )
            neg_loss = F.binary_cross_entropy_with_logits(
                neg_pred, torch.zeros_like(neg_pred)
            )
            loss = pos_loss + neg_loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        return total_loss / max(num_batches, 1)

    @torch.no_grad()
    def evaluate(self, edge_index, sample_size=10000):
        self.model.eval()

        if edge_index.size(1) > sample_size:
            idx = torch.randperm(edge_index.size(1))[:sample_size]
            edge_index = edge_index[:, idx]

        edge_index = edge_index.to(self.device)

        z = self.model(self.x, self.edge_index)

        batch_size = 2048

        pos_preds = []
        for i in range(0, edge_index.size(1), batch_size):
            batch = edge_index[:, i : i + batch_size]
            pos_preds.append(self.model.decode(z, batch).sigmoid().cpu())
        pos_pred = torch.cat(pos_preds)

        neg_preds = []
        for i in range(0, edge_index.size(1), batch_size):
            batch = edge_index[:, i : i + batch_size]
            neg_batch = negative_sampling(
                edge_index=batch,
                num_nodes=self.num_nodes,
                num_neg_samples=batch.size(1),
            )
            neg_preds.append(self.model.decode(z, neg_batch).sigmoid().cpu())
        neg_pred = torch.cat(neg_preds)

        pred = torch.cat([pos_pred, neg_pred]).numpy()
        labels = np.concatenate([np.ones(len(pos_pred)), np.zeros(len(neg_pred))])

        auc = roc_auc_score(labels, pred)
        ap = average_precision_score(labels, pred)

        return auc, ap

    def fit(self, epochs=30, lr=0.0005, weight_decay=1e-5):
        optimizer = torch.optim.AdamW(
            self.model.parameters(), lr=lr, weight_decay=weight_decay
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=5
        )

        best_val_auc = 0
        patience_counter = 0
        patience = 10

        for epoch in range(1, epochs + 1):
            loss = self.train_epoch(optimizer)

            if epoch == 1 or epoch % 10 == 0:
                val_auc, val_ap = self.evaluate(self.val_edges)

                print(
                    f"Epoch {epoch:02d}: "
                    f"Loss={loss:.4f}, Val AUC={val_auc:.4f}, Val AP={val_ap:.4f}"
                )

                scheduler.step(val_auc)

                if val_auc > best_val_auc:
                    best_val_auc = val_auc
                    patience_counter = 0
                    torch.save(
                        {
                            "epoch": epoch,
                            "model_state_dict": self.model.state_dict(),
                            "val_auc": val_auc,
                            "val_ap": val_ap,
                        },
                        "output/best_model.pt",
                    )

                else:
                    patience_counter += 1

                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch}")
                    break

            else:
                print(f"Epoch {epoch:02d}: Loss={loss:.4f}")

        checkpoint = torch.load("output/best_model.pt", weights_only=False)
        self.model.load_state_dict(checkpoint["model_state_dict"])

        test_auc, test_ap = self.evaluate(self.test_edges)

        print(
            f'Best Val AUC: {checkpoint["val_auc"]:.4f} (Epoch {checkpoint["epoch"]})'
        )
        print(f"Test AUC: {test_auc:.4f}")
        print(f"Test AP: {test_ap:.4f}")

        return test_auc, test_ap

In [57]:
data = torch.load("output/data.pt", weights_only=False)

In [58]:
print(f"Nodes: {data.num_nodes:,}")
print(f"Training Edges: {data.edge_index.size(1):,}")
print(f"Validation Edges: {data.val_edges.size(1):,}")
print(f"Test Edges: {data.test_edges.size(1):,}")
print(f"Node Features: {data.x.shape[1]}")

Nodes: 58,929
Training Edges: 9,771,144
Validation Edges: 697,939
Test Edges: 1,395,878
Node Features: 9


In [59]:
model = LightGNN(
    in_channels=data.x.shape[1], hidden_channels=64, out_channels=32, dropout=0.2
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

trainer = Trainer(model, data, batch_size=2048, device=device)

In [61]:
test_auc, test_ap = trainer.fit(epochs=30, lr=0.0005, weight_decay=1e-5)

Epoch 01: Loss=1.3394, Val AUC=0.7170, Val AP=0.6993
Epoch 02: Loss=1.2351
Epoch 03: Loss=1.1906
Epoch 04: Loss=1.1657
Epoch 05: Loss=1.1552
Epoch 06: Loss=1.1483
Epoch 07: Loss=1.1400
Epoch 08: Loss=1.1389
Epoch 09: Loss=1.1360
Epoch 10: Loss=1.1347, Val AUC=0.8349, Val AP=0.8066
Epoch 11: Loss=1.1316
Epoch 12: Loss=1.1295
Epoch 13: Loss=1.1289
Epoch 14: Loss=1.1272
Epoch 15: Loss=1.1240
Epoch 16: Loss=1.1241
Epoch 17: Loss=1.1224
Epoch 18: Loss=1.1204
Epoch 19: Loss=1.1216
Epoch 20: Loss=1.1192, Val AUC=0.8459, Val AP=0.8313
Epoch 21: Loss=1.1146
Epoch 22: Loss=1.1168
Epoch 23: Loss=1.1150
Epoch 24: Loss=1.1125
Epoch 25: Loss=1.1127
Epoch 26: Loss=1.1098
Epoch 27: Loss=1.1086
Epoch 28: Loss=1.1093
Epoch 29: Loss=1.1063
Epoch 30: Loss=1.1085, Val AUC=0.8912, Val AP=0.8666
Best Val AUC: 0.8912 (Epoch 30)
Test AUC: 0.8744
Test AP: 0.8450
